[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/00_smoke_test_t4.ipynb)

# 00 · T4 可行性冒烟测试（助教赛前先跑）/ T4 smoke test

目的：在**真实的 Colab T4** 上确认整套流程能跑通，并把实测数字填进 README 和 `submissions/ref-*.json`。

| # | 测试 | 通过标准 |
|---|------|----------|
| 1 | vLLM 以 fp16 加载 `Qwen3.5-4B-Base` 并生成 | 不报错；记录加载时间、生成 tok/s |
| 2 | lm-eval 跑 GPQA Diamond（`LIMIT=20` 计时） | 有 `results_*.json`；推算整轮耗时 |
| 3 | LoRA SFT 50 步 | 在 `02_sft_dolci_lora.ipynb` 里设 `SMOKE=True`；记录峰值显存、tok/s |
| 4 | HF 后端回退 | 如果 1/2 失败：`BACKEND=hf LIMIT=10` 能出分 |

**Runtime → Change runtime type → T4 GPU。** 每个测试之间最好 **Runtime → Restart session**，因为 vLLM 不会完全释放显存。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆比赛仓库（脚本都在里面） / clone the hackathon repo
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval

In [ ]:
# 评测环境（vLLM + lm-eval）。约 3–5 分钟。装完如果提示重启 runtime，请重启后从下一格继续。
!pip install -q -r requirements-eval.txt
import importlib.metadata as md
for p in ["lm_eval", "vllm", "transformers", "torch"]:
    print(p, md.version(p if p != "lm_eval" else "lm-eval"))

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款，
# 再用有 read（提交模型需要 write）权限的 token 登录。
# 推荐：Colab 左侧 🔑 Secrets 里添加 HF_TOKEN，这里就会自动读取。
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

## Test 1 · vLLM fp16 加载 + 生成

In [ ]:
import time
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen3.5-4B-Base"   # 也测一下 "Qwen/Qwen3.5-4B" 和 "Qwen/Qwen3.5-2B-Base"
t0 = time.time()
llm = LLM(model=MODEL, dtype="float16", max_model_len=8192, gpu_memory_utilization=0.85,
          enforce_eager=True, seed=0)
print(f"load: {time.time() - t0:.0f}s")

prompts = ["Explain why the sky is blue, step by step."] * 32
t1 = time.time()
outs = llm.generate(prompts, SamplingParams(max_tokens=512, temperature=0.6, top_p=0.95))
n_tok = sum(len(o.outputs[0].token_ids) for o in outs)
print(f"generated {n_tok} tokens in {time.time() - t1:.0f}s -> {n_tok / (time.time() - t1):.0f} tok/s (batch 32)")
print(outs[0].outputs[0].text[:500])

如果这里报 GDN / Gated DeltaNet / FlashInfer / `sm90` 相关错误，说明 vLLM 在 T4（sm75）上走不通 → 跳到 Test 4，
并在 README 里把评测后端改成 `hf`，主模型考虑换成 `Qwen3.5-2B-Base`。

**跑完请 Runtime → Restart session，然后重新执行"克隆仓库"那一格（不用重装），再继续 Test 2。**

## Test 2 · lm-eval GPQA Diamond 计时（只跑 20 题）

In [ ]:
%cd /content/GPQA-eval
import time
t0 = time.time()
!LIMIT=20 bash scripts/eval_gpqa.sh Qwen/Qwen3.5-4B-Base results/smoke-base "0" vllm
dt = time.time() - t0
print(f"20 题 × 2 个任务耗时 {dt / 60:.1f} 分钟；完整一轮（198 + 448 题）约 {dt / 40 * 646 / 60:.0f} 分钟 / seed（含模型加载，偏保守）")

In [ ]:
# 看一下汇总（LIMIT 模式的分数没有意义，只检查流程）
!python scripts/aggregate_results.py results/smoke-base --model Qwen/Qwen3.5-4B-Base | head -40

## Test 3 · LoRA 训练速度

在**新的 runtime** 打开 `02_sft_dolci_lora.ipynb`，把 `SMOKE = True`，全部运行，记录最后打印的峰值显存和 tok/s。

## Test 4 · HF 后端回退（仅当 Test 1/2 失败时）

In [ ]:
!pip install -q "lm-eval>=0.4.13" "transformers>=5" accelerate
!LIMIT=10 bash scripts/eval_gpqa.sh Qwen/Qwen3.5-4B-Base results/smoke-hf "0" hf

## 结果记录（填好后更新 README 的"T4 实测"表）

| 项目 | 实测值 |
|------|--------|
| GPU / compute cap | |
| vLLM 版本 / 能否加载 4B-Base | |
| vLLM 生成 tok/s（batch 32） | |
| GPQA 单 seed 完整耗时（vLLM） | |
| HF 后端是否可用 / 耗时 | |
| LoRA SFT 峰值显存 / tok/s（4B, seq 4096） | |
| Base 4 seed：Diamond / Main | |
| 官方 Qwen3.5-4B 4 seed：Diamond / Main | |